# Music Annotation Project V7.8: Independent Score-Level Structural-Safety Validation

**Runtime:** CPU. Select **Runtime > Run all**.

This notebook performs a locked, read-only validation of the V7.7 structural-safety model concept on genuinely independent musical scores. It does not tune thresholds, search for better vetoes, change reconstruction, or authorise automatic acceptance.

## Required independent input

Create:

```text
MusicAnnotationProject/independent_structural_validation_v7_8/
  component_dataset.csv
  score_manifest.csv
```

`component_dataset.csv` must contain the same predictor and structural-outcome columns produced by V7.7, plus `source_group`. `score_manifest.csv` must contain one row per independent score with `source_group`, provenance and exclusion declarations.

## Locked candidate policy

- V7.7 structural probability threshold: **0.98**
- Predeclared evidence-sufficiency hypothesis: review components with **context_mean_intensity > 254.97941844452666**
- The precise intensity cut-off is evaluated only as a frozen retrospective hypothesis. It is not retuned here.
- Failures are defined by the frozen V7.7 structural label.
- Results are reported at component and source-group level.

A validation set is invalid if any score, edition, scan, page, derived crop, mask or clean target contributed to model development, calibration, veto discovery or threshold selection.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy matplotlib scipy scikit-learn


In [ ]:
import json, math, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone
from scipy.stats import beta
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V77_ROOT=ROOT/'structural_risk_calibration_v7_7'
INDEPENDENT_ROOT=ROOT/'independent_structural_validation_v7_8'
INDEPENDENT_COMPONENTS=INDEPENDENT_ROOT/'component_dataset.csv'
SCORE_MANIFEST=INDEPENDENT_ROOT/'score_manifest.csv'
OUTPUT_ROOT=ROOT/'independent_structural_validation_results_v7_8'

LOCKED_SCORE_THRESHOLD=0.98
LOCKED_CONTEXT_VETO_CUTOFF=254.97941844452666
CONFIDENCE_LEVEL=0.95
PRIMARY_MAX_UPPER_RISK=0.005  # Design choice: 0.5% one-sided 95% upper bound.
MIN_INDEPENDENT_SOURCE_GROUPS=10
RANDOM_SEED=42

PREDICTORS=['w','h','area','aspect_ratio','fill_ratio','touches_page_border','context_pixels','context_edge_density','context_dark_density','context_mean_intensity','context_intensity_std','maximum_mask_depth','horizontal_fraction','vertical_fraction','compact_fraction','preserve_fraction','remove_fraction','ambiguous_fraction','reconstruction_dependency_fraction','component_pixels']
FORBIDDEN_OVERLAP_FLAGS=['used_in_v7_7_development','used_in_model_training','used_in_calibration','used_in_veto_discovery']

def latest_complete(root):
    req=['v7_7_manifest.json','tables/asymmetric_component_dataset.csv']
    hits=[p for p in root.rglob('*') if p.is_dir() and all((p/x).exists() for x in req)] if root.exists() else []
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
V77_RUN=latest_complete(V77_ROOT)
if V77_RUN is None: raise RuntimeError('No complete V7.7 development run found.')
if not INDEPENDENT_COMPONENTS.exists(): raise RuntimeError(f'Missing independent component dataset: {INDEPENDENT_COMPONENTS}')
if not SCORE_MANIFEST.exists(): raise RuntimeError(f'Missing independent score manifest: {SCORE_MANIFEST}')
RUN=datetime.now().strftime('%Y%m%d_%H%M%S');OUT=OUTPUT_ROOT/RUN;TABLES=OUT/'tables';CHARTS=OUT/'charts'
for p in [OUT,TABLES,CHARTS]:p.mkdir(parents=True,exist_ok=True)
print('Development source:',V77_RUN);print('Independent input:',INDEPENDENT_ROOT);print('Output:',OUT)


## Validate genuine source-group independence


In [ ]:
dev=pd.read_csv(V77_RUN/'tables'/'asymmetric_component_dataset.csv')
ind=pd.read_csv(INDEPENDENT_COMPONENTS)
manifest=pd.read_csv(SCORE_MANIFEST)
required_ind=set(PREDICTORS+['source_group','page','component','structural_safe'])
missing=required_ind-set(ind.columns)
if missing: raise RuntimeError(f'Independent component dataset missing columns: {sorted(missing)}')
required_manifest={'source_group','score_title','edition_or_source','acquisition_notes','excluded_from_development'}
missing_manifest=required_manifest-set(manifest.columns)
if missing_manifest: raise RuntimeError(f'Score manifest missing columns: {sorted(missing_manifest)}')
if ind.source_group.isna().any() or manifest.source_group.isna().any():raise RuntimeError('Every independent component and score must have source_group.')
if ind.source_group.nunique()<MIN_INDEPENDENT_SOURCE_GROUPS:raise RuntimeError(f'Need at least {MIN_INDEPENDENT_SOURCE_GROUPS} independent source groups; found {ind.source_group.nunique()}.')
if set(ind.source_group)-set(manifest.source_group):raise RuntimeError('Some component source_group values are absent from score_manifest.csv.')
flag=pd.to_numeric(manifest.excluded_from_development,errors='coerce').fillna(0).astype(bool)
if not flag.all():raise RuntimeError('Every score must explicitly be marked excluded_from_development=True.')
for c in FORBIDDEN_OVERLAP_FLAGS:
    if c in manifest.columns and pd.to_numeric(manifest[c],errors='coerce').fillna(0).astype(bool).any():raise RuntimeError(f'Independence violation in manifest column: {c}')
# Optional hash firewall when both datasets carry source hashes.
for c in ['clean_md5','mask_md5','annotated_md5','source_md5']:
    if c in dev.columns and c in ind.columns:
        overlap=set(dev[c].dropna().astype(str)) & set(ind[c].dropna().astype(str))
        if overlap:raise RuntimeError(f'Hash overlap between development and independent data in {c}: {len(overlap)}')
if {'source_group'}.issubset(dev.columns):
    overlap=set(dev.source_group.astype(str)) & set(ind.source_group.astype(str))
    if overlap:raise RuntimeError(f'Source-group overlap with development data: {sorted(overlap)}')
print('Independent source groups:',ind.source_group.nunique(),'pages:',ind.page.nunique(),'components:',len(ind))


## Recreate and freeze the V7.7 development model


In [ ]:
def as_bool(s):
    if s.dtype==bool:return s
    return s.astype(str).str.lower().map({'true':True,'false':False,'1':True,'0':False})
for frame,name in [(dev,'development'),(ind,'independent')]:
    frame['structural_safe']=as_bool(frame.structural_safe)
    if frame.structural_safe.isna().any():raise RuntimeError(f'Unparseable structural_safe values in {name}.')
    for c in PREDICTORS:frame[c]=pd.to_numeric(frame[c],errors='coerce').astype(float)
    if frame[PREDICTORS].isna().any().any():raise RuntimeError(f'Missing/non-numeric predictor values in {name}.')
X=dev[PREDICTORS];y=dev.structural_safe.astype(int);groups=dev.page.astype(str)
base=RandomForestClassifier(n_estimators=800,min_samples_leaf=4,max_features='sqrt',class_weight='balanced_subsample',random_state=RANDOM_SEED,n_jobs=-1)
# OOF predictions are used only to fit the locked monotonic calibrator without in-sample leakage.
folds=min(10,groups.nunique());cv=GroupKFold(n_splits=folds)
oof=cross_val_predict(base,X,y,groups=groups,cv=cv,method='predict_proba',n_jobs=-1)[:,1]
cal=IsotonicRegression(out_of_bounds='clip').fit(oof,y)
base.fit(X,y)
ind_raw=base.predict_proba(ind[PREDICTORS])[:,1]
ind['prob_structural_safe']=cal.transform(ind_raw)
print('Development OOF ROC AUC:',roc_auc_score(y,oof));print('Development OOF AP:',average_precision_score(y,oof))


## Apply only the locked candidate policy


In [ ]:
ind['score_eligible']=ind.prob_structural_safe>=LOCKED_SCORE_THRESHOLD
ind['context_veto']=ind.context_mean_intensity>LOCKED_CONTEXT_VETO_CUTOFF
ind['auto_candidate']=ind.score_eligible & ~ind.context_veto
ind['structural_failure']=~ind.structural_safe
ind['auto_candidate_failure']=ind.auto_candidate & ind.structural_failure
ind.to_csv(TABLES/'independent_component_predictions.csv',index=False)
print('Score eligible:',int(ind.score_eligible.sum()));print('Vetoed:',int((ind.score_eligible&ind.context_veto).sum()));print('Auto candidates:',int(ind.auto_candidate.sum()));print('Failures:',int(ind.auto_candidate_failure.sum()))


## Exact binomial risk bounds and predeclared acceptance gate


In [ ]:
def clopper_pearson_upper(failures,n,confidence=.95):
    if n==0:return np.nan
    alpha=1-confidence
    return float(beta.ppf(1-alpha,failures+1,n-failures)) if failures<n else 1.0

def zero_failure_n(max_risk,confidence=.95):
    return int(math.ceil(math.log(1-confidence)/math.log(1-max_risk)))

n=int(ind.auto_candidate.sum());fail=int(ind.auto_candidate_failure.sum())
upper=clopper_pearson_upper(fail,n,CONFIDENCE_LEVEL)
required_n=zero_failure_n(PRIMARY_MAX_UPPER_RISK,CONFIDENCE_LEVEL)
primary_pass=bool(fail==0 and n>=required_n and upper<=PRIMARY_MAX_UPPER_RISK)
summary=pd.DataFrame([{'confidence_level':CONFIDENCE_LEVEL,'maximum_acceptable_upper_risk':PRIMARY_MAX_UPPER_RISK,'auto_candidates':n,'failures':fail,'observed_failure_rate':fail/n if n else np.nan,'exact_one_sided_upper_bound':upper,'zero_failure_sample_required':required_n,'source_groups':ind.loc[ind.auto_candidate,'source_group'].nunique(),'primary_gate_pass':primary_pass}])
summary.to_csv(TABLES/'primary_validation_gate.csv',index=False)
# Planning table
planning=[]
for confidence in [.95,.99]:
    for risk in [.02,.01,.005,.0025,.001,.0005]:
        planning.append({'confidence_level':confidence,'target_upper_risk':risk,'zero_failure_components_required':zero_failure_n(risk,confidence)})
pd.DataFrame(planning).to_csv(TABLES/'zero_failure_sample_size_planning.csv',index=False)
display(summary);display(pd.DataFrame(planning))


## Source-group robustness


In [ ]:
rows=[]
for source,g in ind.groupby('source_group'):
    sel=g.auto_candidate;n=int(sel.sum());fail=int((sel&g.structural_failure).sum())
    rows.append({'source_group':source,'pages':g.page.nunique(),'components':len(g),'score_eligible':int(g.score_eligible.sum()),'vetoed_score_eligible':int((g.score_eligible&g.context_veto).sum()),'auto_candidates':n,'failures':fail,'observed_failure_rate':fail/n if n else np.nan,'upper95':clopper_pearson_upper(fail,n,.95)})
by_source=pd.DataFrame(rows).sort_values(['failures','auto_candidates'],ascending=[False,False]);by_source.to_csv(TABLES/'validation_by_source_group.csv',index=False);display(by_source)
# Leave-one-source-out aggregate sensitivity
loo=[]
for source in sorted(ind.source_group.unique()):
    g=ind[ind.source_group!=source];n=int(g.auto_candidate.sum());f=int(g.auto_candidate_failure.sum())
    loo.append({'excluded_source_group':source,'auto_candidates':n,'failures':f,'upper95':clopper_pearson_upper(f,n,.95)})
pd.DataFrame(loo).to_csv(TABLES/'leave_one_source_group_out.csv',index=False)


## Stratified reporting


In [ ]:
# Report available locked strata without inventing missing labels.
strata=[c for c in ['region','component_category','notation_proximity_class','mask_content_class'] if c in ind.columns]
records=[]
for c in strata:
    for value,g in ind.groupby(c,dropna=False):
        n=int(g.auto_candidate.sum());f=int(g.auto_candidate_failure.sum())
        records.append({'stratum_variable':c,'stratum_value':value,'components':len(g),'auto_candidates':n,'failures':f,'observed_failure_rate':f/n if n else np.nan,'upper95':clopper_pearson_upper(f,n,.95)})
strata_df=pd.DataFrame(records)
strata_df.to_csv(TABLES/'validation_by_available_stratum.csv',index=False)
print('Available strata:',strata if strata else 'none supplied')


## Charts and immutable validation record


In [ ]:
# Predicted score distribution by outcome.
plt.figure(figsize=(10,6))
for label,g in ind.groupby('structural_safe'):
    plt.hist(g.prob_structural_safe,bins=30,alpha=.55,label='structural_safe' if label else 'structural_unsafe')
plt.axvline(LOCKED_SCORE_THRESHOLD,color='black',linestyle='--',label='locked threshold');plt.xlabel('Locked calibrated structural-safety probability');plt.ylabel('Components');plt.legend();plt.tight_layout();plt.savefig(CHARTS/'independent_probability_distribution.png',dpi=180);plt.close()
# Coverage by source group.
plot=by_source.sort_values('auto_candidates');plt.figure(figsize=(12,max(6,.35*len(plot))));plt.barh(plot.source_group.astype(str),plot.auto_candidates,color=np.where(plot.failures>0,'crimson','steelblue'));plt.xlabel('Auto-candidate components');plt.title('Independent auto-candidate coverage by source group');plt.tight_layout();plt.savefig(CHARTS/'auto_candidate_coverage_by_source.png',dpi=180);plt.close()
manifest_out={'created_utc':datetime.now(timezone.utc).isoformat(),'development_v7_7_run':str(V77_RUN),'independent_component_dataset':str(INDEPENDENT_COMPONENTS),'score_manifest':str(SCORE_MANIFEST),'locked_policy':{'score_threshold':LOCKED_SCORE_THRESHOLD,'context_mean_intensity_veto_above':LOCKED_CONTEXT_VETO_CUTOFF},'components':int(len(ind)),'source_groups':int(ind.source_group.nunique()),'pages':int(ind.page.nunique()),'auto_candidates':n,'failures':fail,'confidence_level':CONFIDENCE_LEVEL,'exact_one_sided_upper_bound':upper,'predeclared_maximum_upper_risk':PRIMARY_MAX_UPPER_RISK,'zero_failure_sample_required':required_n,'primary_gate_pass':primary_pass,'no_threshold_or_veto_tuning_performed':True,'production_authorisation':False}
(OUT/'v7_8_manifest.json').write_text(json.dumps(manifest_out,indent=2))
text=chr(10).join(['V7.8 INDEPENDENT SCORE-LEVEL STRUCTURAL-SAFETY VALIDATION','',f'Independent source groups: {ind.source_group.nunique()}',f'Independent pages: {ind.page.nunique()}',f'Independent components: {len(ind)}',f'Auto candidates: {n}',f'Structural failures: {fail}',f'Observed failure rate: {fail/n if n else float("nan"):.6f}',f'Exact one-sided {CONFIDENCE_LEVEL:.0%} upper risk bound: {upper:.6f}',f'Predeclared maximum upper risk: {PRIMARY_MAX_UPPER_RISK:.6f}',f'Zero-failure sample required: {required_n}',f'PRIMARY GATE PASS: {primary_pass}','','This notebook does not authorise production auto-acceptance. Annotation confidence remains a separate required axis.'])
(OUT/'executive_summary.txt').write_text(text);print(text);print('Output:',OUT)
